# Import libraries & define paths

In [50]:
from pathlib import Path
import pandas as pd
import requests

In [51]:
# -----------------------------
# Project paths
# -----------------------------


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for path in [RAW_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project
Raw data directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw
Processed data directory: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed


# 01 Data Collection

This notebook collects the first official datasets for the AI Literacy Gap Index.  
The project starts with Eurostat's regional digital skills dataset because digital skills are the most direct available proxy for AI literacy readiness at NUTS-1 level.

The first goal is to download the Eurostat data inventory, identify the relevant dataset, and save the raw data locally so that later notebooks can work from reproducible local files.

In [52]:
# -----------------------------
# Eurostat inventory download
# -----------------------------

EUROSTAT_INVENTORY_URL = "https://ec.europa.eu/eurostat/api/dissemination/files/inventory?type=data&lang=en"

inventory_path = RAW_DIR / "eurostat_inventory.tsv"

response = requests.get(EUROSTAT_INVENTORY_URL, timeout=60)
response.raise_for_status()

inventory_path.write_bytes(response.content)

eurostat_inventory = pd.read_csv(inventory_path, sep="\t")

print("Inventory shape:", eurostat_inventory.shape)
display(eurostat_inventory.head())
display(eurostat_inventory.columns)

Inventory shape: (8218, 10)


,Code,Type,Source dataset,Last data change,Last structural change,Data download url (tsv),Data download url (csv),Data download url (sdmx),Data structure download url,Open in Data Browser url
0,AACT_ALI01,DATASET,-,2026-05-13T11:00:00+0200,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
1,AACT_ALI01_R,DATASET,-,2026-03-24T11:00:00+0100,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
2,AACT_ALI02,DATASET,-,2026-05-13T11:00:00+0200,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
3,AACT_ALI02_R,DATASET,-,2026-03-24T11:00:00+0100,2026-03-24T11:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...
4,AACT_EAA01,DATASET,-,2026-05-13T11:00:00+0200,2026-03-23T23:00:00+0100,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/api/disseminatio...,https://ec.europa.eu/eurostat/databrowser/prod...


Index(['Code', 'Type', 'Source dataset', 'Last data change',
       'Last structural change', 'Data download url (tsv)',
       'Data download url (csv)', 'Data download url (sdmx)',
       'Data structure download url', 'Open in Data Browser url'],
      dtype='str')

## Download core dataset: regional digital skills

The first core dataset for the index is Eurostat's regional digital skills dataset.  
This dataset is used as the main proxy for digital readiness, which is one of the central components of the AI Literacy Gap Index.

Before downloading the data, I check whether the dataset exists in the Eurostat inventory and retrieve its official download URL from there.

In [ ]:
# -----------------------------
# Find Eurostat digital skills dataset
# -----------------------------

TARGET_DATASET = "isoc_r_dskl_i"

dataset_match = eurostat_inventory[
    eurostat_inventory["Code"].str.lower() == TARGET_DATASET.lower()
]

if dataset_match.empty:
    raise ValueError(f"Dataset {TARGET_DATASET} was not found in the Eurostat inventory.")

display(dataset_match.T)

# Get official TSV download URL from the inventory
download_url = dataset_match["Data download url (tsv)"].iloc[0]

print("Dataset code:", TARGET_DATASET)
print("Download URL:", download_url)

# -----------------------------
# Download raw dataset
# -----------------------------

raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

response = requests.get(download_url, timeout=120)
response.raise_for_status()

raw_dataset_path.write_bytes(response.content)

print(f"Saved raw dataset to: {raw_dataset_path}")

# Load a first preview
digital_skills_raw = pd.read_csv(raw_dataset_path, sep="\t")

print("Raw dataset shape:", digital_skills_raw.shape)
display(digital_skills_raw.head())
display(digital_skills_raw.columns)

,4289
Code,ISOC_R_DSKL_I
Type,DATASET
Source dataset,-
Last data change,2026-04-17T11:00:00+0200
Last structural change,2026-04-17T11:00:00+0200
Data download url (tsv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (csv),https://ec.europa.eu/eurostat/api/disseminatio...
Data download url (sdmx),https://ec.europa.eu/eurostat/api/disseminatio...
Data structure download url,https://ec.europa.eu/eurostat/api/disseminatio...
Open in Data Browser url,https://ec.europa.eu/eurostat/databrowser/prod...


Dataset code: isoc_r_dskl_i
Download URL: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/ISOC_R_DSKL_I/?format=TSV
Saved raw dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\raw\isoc_r_dskl_i.tsv
Raw dataset shape: (5814, 2)


,"freq,indic_is,unit,geo\TIME_PERIOD",2025
0,"A,I_DSK2_AB,PC_IND,AL",8.07
1,"A,I_DSK2_AB,PC_IND,AT",34.26
2,"A,I_DSK2_AB,PC_IND,AT1",36.81
3,"A,I_DSK2_AB,PC_IND,AT2",31.11
4,"A,I_DSK2_AB,PC_IND,AT3",32.85


Index(['freq,indic_is,unit,geo\TIME_PERIOD', '2025 '], dtype='str')

## Reshape the digital skills dataset

The raw Eurostat file stores several metadata fields in one combined column and the year as a separate value column.  
To make the data usable for analysis, I reshape it into a tidy long format with one row per region, indicator, unit, and year.

In [54]:
# -----------------------------
# Reshape Eurostat compact TSV format
# -----------------------------

# Reload raw data if needed
TARGET_DATASET = "isoc_r_dskl_i"
raw_dataset_path = RAW_DIR / f"{TARGET_DATASET}.tsv"

digital_skills_raw = pd.read_csv(raw_dataset_path, sep="\t")

# Clean column names
digital_skills_raw.columns = digital_skills_raw.columns.str.strip()

# Identify the combined dimension column
dimension_col = [col for col in digital_skills_raw.columns if "\\" in col][0]

# The part before "\TIME_PERIOD" contains the dimension names
dimension_names = dimension_col.split("\\")[0].split(",")

print("Dimension column:", dimension_col)
print("Detected dimensions:", dimension_names)

# Split combined dimension values into separate columns
digital_skills_split = digital_skills_raw[dimension_col].str.split(",", expand=True)
digital_skills_split.columns = dimension_names

# Add year columns
value_cols = [col for col in digital_skills_raw.columns if col != dimension_col]

digital_skills_wide = pd.concat(
    [digital_skills_split, digital_skills_raw[value_cols]],
    axis=1
)

# Reshape from wide to long format
digital_skills_long = digital_skills_wide.melt(
    id_vars=dimension_names,
    value_vars=value_cols,
    var_name="year",
    value_name="value_raw"
)

# Clean year and value columns
digital_skills_long["year"] = digital_skills_long["year"].astype(str).str.strip().astype(int)

# Eurostat values can contain flags after the numeric value.
# This extracts the numeric part and keeps missing values as NaN.
digital_skills_long["value"] = (
    digital_skills_long["value_raw"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")
    .astype(float)
)

# Clean text columns
for col in dimension_names:
    digital_skills_long[col] = digital_skills_long[col].astype(str).str.strip()

# Basic preview
print("Tidy dataset shape:", digital_skills_long.shape)
display(digital_skills_long.head())

print("Available years:")
display(sorted(digital_skills_long["year"].dropna().unique()))

print("Available units:")
display(digital_skills_long["unit"].value_counts())

print("Number of regions:", digital_skills_long["geo"].nunique())
print("Number of indicators:", digital_skills_long["indic_is"].nunique())

Dimension column: freq,indic_is,unit,geo\TIME_PERIOD
Detected dimensions: ['freq', 'indic_is', 'unit', 'geo']
Tidy dataset shape: (5814, 7)


,freq,indic_is,unit,geo,year,value_raw,value
0,A,I_DSK2_AB,PC_IND,AL,2025,8.07,8.07
1,A,I_DSK2_AB,PC_IND,AT,2025,34.26,34.26
2,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81
3,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11
4,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85


Available years:


[np.int64(2025)]

Available units:


unit
PC_IND        2969
PC_IND_IU3    2845
Name: count, dtype: int64

Number of regions: 124
Number of indicators: 24


## Inspect available digital skills indicators

Before selecting variables for the index, I inspect the available digital skills indicators, units, regions, and missing values.  
This step helps separate exploratory understanding from final indicator selection.

In [55]:
# -----------------------------
# Inspect available indicators and data coverage
# -----------------------------

print("Dataset shape:", digital_skills_long.shape)
print("Years:", sorted(digital_skills_long["year"].unique()))
print("Units:", sorted(digital_skills_long["unit"].unique()))
print("Number of geo codes:", digital_skills_long["geo"].nunique())
print("Number of indicators:", digital_skills_long["indic_is"].nunique())

# Indicator-level overview
indicator_overview = (
    digital_skills_long
    .groupby(["indic_is", "unit"], as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum()),
        min_value=("value", "min"),
        median_value=("value", "median"),
        max_value=("value", "max")
    )
    .sort_values(["indic_is", "unit"])
)

display(indicator_overview)

# Region-level missingness overview
region_overview = (
    digital_skills_long
    .groupby("geo", as_index=False)
    .agg(
        n_rows=("value", "size"),
        n_non_missing=("value", "count"),
        n_missing=("value", lambda x: x.isna().sum())
    )
    .assign(missing_share=lambda df: df["n_missing"] / df["n_rows"])
    .sort_values("missing_share", ascending=False)
)

display(region_overview.head(20))

# Save tidy interim dataset for later notebooks
interim_path = PROCESSED_DIR / "digital_skills_nuts1_tidy.csv"
digital_skills_long.to_csv(interim_path, index=False)

print(f"Saved tidy digital skills dataset to: {interim_path}")

Dataset shape: (5814, 7)
Years: [np.int64(2025)]
Units: ['PC_IND', 'PC_IND_IU3']
Number of geo codes: 124
Number of indicators: 24


,indic_is,unit,n_rows,n_non_missing,n_missing,min_value,median_value,max_value
0,I_DSK2_AB,PC_IND,124,124,0,3.70,27.960,59.42
1,I_DSK2_AB,PC_IND_IU3,124,124,0,4.27,30.650,59.56
2,I_DSK2_B,PC_IND,124,124,0,13.41,28.545,66.44
3,I_DSK2_B,PC_IND_IU3,124,124,0,15.93,30.210,67.48
4,I_DSK2_BAB,PC_IND,124,124,0,18.56,59.570,85.17
5,I_DSK2_BAB,PC_IND_IU3,124,124,0,21.63,63.335,85.36
6,I_DSK2_CC_AB,PC_IND,124,124,0,71.29,85.990,99.42
7,I_DSK2_CC_AB,PC_IND_IU3,124,124,0,83.23,93.600,99.56
8,I_DSK2_CC_B,PC_IND,124,124,0,0.44,4.615,11.97
9,I_DSK2_CC_B,PC_IND_IU3,124,124,0,0.44,4.880,12.94


,geo,n_rows,n_non_missing,n_missing,missing_share
20,DE4,47,45,2,0.042553
23,DE8,47,45,2,0.042553
27,DEE,47,45,2,0.042553
19,DE3,47,45,2,0.042553
24,DE9,47,45,2,0.042553
33,EL4,47,45,2,0.042553
44,FI,47,45,2,0.042553
80,NL2,47,45,2,0.042553
83,NO,47,45,2,0.042553
82,NL4,47,45,2,0.042553


Saved tidy digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_tidy.csv


## Decode Eurostat indicator labels

The raw dataset uses Eurostat technical codes for indicators, units, and regions.  
Before selecting variables for the AI Literacy Gap Index, I decode these codes into readable labels so that indicator choices can be made transparently.

In [ ]:
# -----------------------------
# Download Eurostat JSON metadata for labels
# -----------------------------

metadata_url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{TARGET_DATASET}"

params = {
    "lang": "en",
    "time": "2025"
}

response = requests.get(metadata_url, params=params, timeout=120)
response.raise_for_status()

digital_skills_metadata = response.json()

print("Metadata keys:", digital_skills_metadata.keys())
print("Available dimensions:", digital_skills_metadata.get("id", []))


# -----------------------------
# Helper function to extract dimension labels
# -----------------------------
def extract_dimension_labels(metadata, dimension_name):
    """
    Extracts Eurostat code-label mappings from the JSON metadata.
    """
    dimension = metadata["dimension"][dimension_name]
    category = dimension["category"]

    labels = category.get("label", {})
    index = category.get("index", {})

    # Prefer index keys because they define the available codes
    codes = list(index.keys()) if isinstance(index, dict) else list(labels.keys())

    label_table = pd.DataFrame({
        dimension_name: codes,
        f"{dimension_name}_label": [labels.get(code, code) for code in codes]
    })

    return label_table


# -----------------------------
# Extract labels for relevant dimensions
# -----------------------------

indic_labels = extract_dimension_labels(digital_skills_metadata, "indic_is")
unit_labels = extract_dimension_labels(digital_skills_metadata, "unit")
geo_labels = extract_dimension_labels(digital_skills_metadata, "geo")

print("Indicator labels:")
display(indic_labels)

print("Unit labels:")
display(unit_labels)

print("Geo labels preview:")
display(geo_labels.head(20))


# -----------------------------
# Merge labels into tidy dataset
# -----------------------------

digital_skills_labeled = (
    digital_skills_long
    .merge(indic_labels, on="indic_is", how="left")
    .merge(unit_labels, on="unit", how="left")
    .merge(geo_labels, on="geo", how="left")
)

display(digital_skills_labeled.head())

# Save labeled dataset
labeled_path = PROCESSED_DIR / "digital_skills_nuts1_labeled.csv"
digital_skills_labeled.to_csv(labeled_path, index=False)

print(f"Saved labeled digital skills dataset to: {labeled_path}")

Metadata keys: dict_keys(['version', 'class', 'label', 'source', 'updated', 'value', 'status', 'id', 'size', 'dimension', 'extension'])
Available dimensions: ['freq', 'indic_is', 'unit', 'geo', 'time']
Indicator labels:


,indic_is,indic_is_label
0,I_DSK2_IC_S,Individuals with online information and commun...
1,I_DSK2_DCC_BAB,Individuals with basic or above basic digital ...
2,I_DSK2_DCC_AB,Individuals with above basic digital content c...
3,I_DSK2_DCC_B,Individuals with basic digital content creatio...
4,I_DSK2_SF_BAB,Individuals with basic or above basic safety s...
5,I_DSK2_SF_AB,Individuals with above basic safety skills
6,I_DSK2_SF_B,Individuals with basic safety skills
7,I_DSK2_PS_BAB,Individuals with basic or above basic problem ...
8,I_DSK2_PS_AB,Individuals with above basic problem solving s...
9,I_DSK2_PS_B,Individuals with basic problem solving skills


Unit labels:


,unit,unit_label
0,PC_IND,Percentage of individuals
1,PC_IND_IU3,Percentage of individuals who used internet in...


Geo labels preview:


,geo,geo_label
0,BE,Belgium
1,BE1,Région de Bruxelles-Capitale/Brussels Hoofdste...
2,BE2,Vlaams Gewest
3,BE3,Région wallonne
4,BG,Bulgaria
5,BG3,Severna i Yugoiztochna Bulgaria
6,BG4,Yugozapadna i Yuzhna tsentralna Bulgaria
7,CZ,Czechia
8,DK,Denmark
9,DE,Germany


,freq,indic_is,unit,geo,year,value_raw,value,indic_is_label,unit_label,geo_label
0,A,I_DSK2_AB,PC_IND,AL,2025,8.07,8.07,Individuals with above basic overall digital s...,Percentage of individuals,Albania
1,A,I_DSK2_AB,PC_IND,AT,2025,34.26,34.26,Individuals with above basic overall digital s...,Percentage of individuals,Austria
2,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81,Individuals with above basic overall digital s...,Percentage of individuals,Ostösterreich
3,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11,Individuals with above basic overall digital s...,Percentage of individuals,Südösterreich
4,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85,Individuals with above basic overall digital s...,Percentage of individuals,Westösterreich


Saved labeled digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_labeled.csv


## Filter to official NUTS-1 regions

The Eurostat digital skills dataset contains both country-level and regional geo codes.  
For this project, the unit of analysis is NUTS-1 regions, so I match the dataset against the official NUTS-1 classification and keep only valid NUTS-1 geo codes.

This avoids relying on simple code-length rules, because some countries can also be represented as a single NUTS-1 region.

In [57]:
# -----------------------------
# Download official NUTS-1 classification from GISCO
# -----------------------------

import json

NUTS1_GEOJSON_URL = (
    "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
    "NUTS_RG_01M_2024_4326_LEVL_1.geojson"
)

nuts1_geojson_path = RAW_DIR / "NUTS_RG_01M_2024_4326_LEVL_1.geojson"

response = requests.get(NUTS1_GEOJSON_URL, timeout=120)
response.raise_for_status()

nuts1_geojson_path.write_bytes(response.content)

with open(nuts1_geojson_path, "r", encoding="utf-8") as f:
    nuts1_geojson = json.load(f)

# Extract NUTS-1 metadata from GeoJSON properties
nuts1_lookup = pd.DataFrame([
    feature["properties"]
    for feature in nuts1_geojson["features"]
])

# Keep only relevant columns if available
available_cols = [col for col in ["NUTS_ID", "NAME_LATN", "CNTR_CODE", "LEVL_CODE"] if col in nuts1_lookup.columns]
nuts1_lookup = nuts1_lookup[available_cols].drop_duplicates()

nuts1_lookup = nuts1_lookup.rename(columns={
    "NUTS_ID": "geo",
    "NAME_LATN": "nuts1_name",
    "CNTR_CODE": "country_code",
    "LEVL_CODE": "nuts_level"
})

print("Official NUTS-1 regions:", nuts1_lookup["geo"].nunique())
display(nuts1_lookup.head())


# -----------------------------
# Filter digital skills data to official NUTS-1 regions
# -----------------------------

digital_skills_nuts1 = digital_skills_labeled.merge(
    nuts1_lookup,
    on="geo",
    how="inner"
)

excluded_geo_codes = sorted(
    set(digital_skills_labeled["geo"].unique()) - set(digital_skills_nuts1["geo"].unique())
)

print("Original geo codes:", digital_skills_labeled["geo"].nunique())
print("Matched NUTS-1 geo codes:", digital_skills_nuts1["geo"].nunique())
print("Excluded geo codes:", len(excluded_geo_codes))
print("Examples of excluded geo codes:")
display(excluded_geo_codes[:30])

print("Filtered NUTS-1 dataset shape:", digital_skills_nuts1.shape)
display(digital_skills_nuts1.head())


# -----------------------------
# Save filtered NUTS-1 dataset
# -----------------------------

nuts1_path = PROCESSED_DIR / "digital_skills_nuts1_labeled_filtered.csv"
digital_skills_nuts1.to_csv(nuts1_path, index=False)

nuts1_lookup_path = PROCESSED_DIR / "nuts1_lookup.csv"
nuts1_lookup.to_csv(nuts1_lookup_path, index=False)

print(f"Saved filtered NUTS-1 digital skills dataset to: {nuts1_path}")
print(f"Saved NUTS-1 lookup table to: {nuts1_lookup_path}")

Official NUTS-1 regions: 115


,geo,nuts1_name,country_code,nuts_level
0,DEF,Schleswig-Holstein,DE,1
1,DE2,Bayern,DE,1
2,DE3,Berlin,DE,1
3,DEG,Thüringen,DE,1
4,EL4,"Nisia Aigaiou, Kriti",EL,1


Original geo codes: 124
Matched NUTS-1 geo codes: 88
Excluded geo codes: 36
Examples of excluded geo codes:


['AL',
 'AT',
 'BA',
 'BE',
 'BG',
 'CH',
 'CY',
 'CZ',
 'DE',
 'DK',
 'EE',
 'EL',
 'ES',
 'FI',
 'FR',
 'HR',
 'HU',
 'IE',
 'IT',
 'LT',
 'LU',
 'LV',
 'ME',
 'MK',
 'MT',
 'NL',
 'NO',
 'PL',
 'PT',
 'RO']

Filtered NUTS-1 dataset shape: (4126, 13)


,freq,indic_is,unit,geo,year,value_raw,value,indic_is_label,unit_label,geo_label,nuts1_name,country_code,nuts_level
0,A,I_DSK2_AB,PC_IND,AT1,2025,36.81,36.81,Individuals with above basic overall digital s...,Percentage of individuals,Ostösterreich,Ostösterreich,AT,1
1,A,I_DSK2_AB,PC_IND,AT2,2025,31.11,31.11,Individuals with above basic overall digital s...,Percentage of individuals,Südösterreich,Südösterreich,AT,1
2,A,I_DSK2_AB,PC_IND,AT3,2025,32.85,32.85,Individuals with above basic overall digital s...,Percentage of individuals,Westösterreich,Westösterreich,AT,1
3,A,I_DSK2_AB,PC_IND,BE1,2025,35.82,35.82,Individuals with above basic overall digital s...,Percentage of individuals,Région de Bruxelles-Capitale/Brussels Hoofdste...,Région de Bruxelles-Capitale/Brussels Hoofdste...,BE,1
4,A,I_DSK2_AB,PC_IND,BE2,2025,29.59,29.59,Individuals with above basic overall digital s...,Percentage of individuals,Vlaams Gewest,Vlaams Gewest,BE,1


Saved filtered NUTS-1 digital skills dataset to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\digital_skills_nuts1_labeled_filtered.csv
Saved NUTS-1 lookup table to: c:\Users\Lu\OneDrive\ToU\chl_data_science_project\data\processed\nuts1_lookup.csv
